In [ ]:
# =========================================
# INSTALL
# =========================================
!pip install networkx scikit-learn


# =========================================
# IMPORTS
# =========================================
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import KFold # Added for cross-validation


# =========================================
# LOAD DATA
# =========================================
print("\n[INFO] Loading data...")
df = pd.read_csv("processed_can_data.csv")
ref_df = pd.read_csv("normal_reference_values.csv")

df["CAN_ID"] = df["CAN_ID"].astype(str).str.strip()
df["Flag"] = df["Flag"].astype(str).str.strip()

print("[INFO] Data Loaded:", df.shape)


# =========================================
# PARAMETERS
# =========================================
window_sizes = [50, 100, 500, 1000]
alpha = 0.02

paper_thresholds = {
    50: {"gamma1": 5.0, "gamma2": 5.0},
    100: {"gamma1": 3.0, "gamma2": 4.3},
    500: {"gamma1": 1.8, "gamma2": 2.5},
    1000: {"gamma1": 1.1, "gamma2": 1.6}
}

gamma3_values = {
    50: 4.4,
    100: 2.8,
    500: 3.2,
    1000: 3.7
}


# =========================================
# SPLIT FIRST (NO LEAKAGE)
# =========================================
print("\n[INFO] Splitting dataset...")
split_idx = int(0.7 * len(df))

train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

print("[INFO] Train:", train_df.shape)
print("[INFO] Test :", test_df.shape)


# =========================================
# ENTROPY
# =========================================
def calculate_entropy(values):
    counts = Counter(values)
    total = len(values)
    probs = [c/total for c in counts.values()]
    return -sum(p*np.log2(p) for p in probs if p > 0)


# =========================================
# GRAPH FEATURES
# =========================================
def build_graph(can_ids):
    G = nx.DiGraph()
    for i in range(len(can_ids)-1):
        G.add_edge(can_ids[i], can_ids[i+1])
    return G


def graph_features(can_ids):
    G = build_graph(can_ids)

    nodes = G.number_of_nodes()
    edges = G.number_of_edges()

    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())

    max_in = max(in_deg.values()) if in_deg else 0
    min_in = min(in_deg.values()) if in_deg else 0
    max_out = max(out_deg.values()) if out_deg else 0
    min_out = min(out_deg.values()) if out_deg else 0

    pr = nx.pagerank(G) if nodes > 0 else {}
    pr_vals = list(pr.values()) if pr else [0]

    pr_max = max(pr_vals)
    pr_min = min(pr_vals)
    pr_median = np.median(pr_vals)

    return [
        nodes, edges,
        max_in, max_out,
        min_in, min_out,
        pr_max, pr_min, pr_median
    ]


# =========================================
# STATISTICAL FEATURES
# =========================================
def stat_features(window, ref, gamma1, gamma2, gamma3):
    can_ids = window["CAN_ID"].tolist()

    freq = Counter(can_ids).values()

    mean = np.mean(list(freq))
    std = np.std(list(freq))
    entropy = calculate_entropy(can_ids)

    mean_score = abs(mean - ref["mu_a"]) / (ref["sigma_a"] + 1e-9)
    std_score  = abs(std  - ref["mu_s"]) / (ref["sigma_s"] + 1e-9)
    ent_score  = abs(entropy - ref["mu_e"]) / (ref["sigma_e"] + 1e-9)

    v1 = int(mean_score > gamma1)
    v2 = int(std_score > gamma2)
    v3 = int(ent_score > gamma3)

    score = v1 + v2 + v3

    return mean, std, entropy, score


# =========================================
# MAIN LOOP PER WINDOW SIZE
# =========================================
for w in window_sizes:

    print("\n" + "="*60)
    print(f"[INFO] PROCESSING WINDOW SIZE = {w}")
    print("="*60)

    ref = ref_df[ref_df["Window_Size"] == w].iloc[0]

    gamma1 = paper_thresholds[w]["gamma1"]
    gamma2 = paper_thresholds[w]["gamma2"]
    gamma3 = gamma3_values[w]

    print(f"[INFO] gamma1={gamma1}, gamma2={gamma2}, gamma3={gamma3}")

    # -------------------------------
    # DATASET CREATION
    # -------------------------------
    def create_dataset(data, name):
        print(f"\n[INFO] Creating {name} dataset...")

        X = []
        y = []

        total_windows = len(data) // w
        print(f"[INFO] Total windows: {total_windows}")

        counter = 0

        for start in range(0, len(data), w):
            end = start + w
            if end > len(data):
                break

            window = data.iloc[start:end]

            attack_ratio = (window["Flag"] == "T").sum() / w
            label = 1 if attack_ratio >= alpha else 0

            mean, std, entropy, score = stat_features(
                window, ref, gamma1, gamma2, gamma3
            )

            g_feats = graph_features(window["CAN_ID"].tolist())

            features = [mean, std, entropy, score] + g_feats

            X.append(features)
            y.append(label)

            counter += 1

            if counter % 5000 == 0:
                print(f"[INFO] {name}: {counter}/{total_windows} processed")

        print(f"[INFO] Finished {name} dataset")

        return np.array(X), np.array(y)

    # Create datasets
    X_train_full, y_train_full = create_dataset(train_df, "TRAIN_FULL") # Renamed for clarity
    X_test, y_test   = create_dataset(test_df, "TEST")

    print(f"[INFO] Train (full) shape: {X_train_full.shape}") # Adjusted print
    print(f"[INFO] Test shape : {X_test.shape}")

    # -------------------------------
    # CROSS-VALIDATION
    # -------------------------------
    print("\n[INFO] Performing K-Fold Cross-Validation...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42) # 5 folds, shuffled for robustness

    cv_accuracies = []
    cv_precisions = []
    cv_recal_scores = []
    cv_f1_scores = []

    fold_num = 1
    for train_index, val_index in kf.split(X_train_full):
        print(f"[INFO] Processing Fold {fold_num}...")
        X_train_fold, X_val_fold = X_train_full[train_index], X_train_full[val_index]
        y_train_fold, y_val_fold = y_train_full[train_index], y_train_full[val_index]

        model_cv = GaussianNB()
        model_cv.fit(X_train_fold, y_train_fold)
        y_pred_cv = model_cv.predict(X_val_fold)

        report_cv = classification_report(y_val_fold, y_pred_cv, output_dict=True, zero_division=0)

        cv_accuracies.append(report_cv['accuracy'])
        cv_precisions.append(report_cv['1']['precision'] if '1' in report_cv else 0.0)
        cv_recal_scores.append(report_cv['1']['recall'] if '1' in report_cv else 0.0)
        cv_f1_scores.append(report_cv['1']['f1-score'] if '1' in report_cv else 0.0)

        fold_num += 1

    print("\n[RESULT] Cross-Validation Results (Attack Class - 1):")
    print(f"  Average Accuracy:  {np.mean(cv_accuracies):.4f} (+/- {np.std(cv_accuracies):.4f})")
    print(f"  Average Precision: {np.mean(cv_precisions):.4f} (+/- {np.std(cv_precisions):.4f})")
    print(f"  Average Recall:    {np.mean(cv_recal_scores):.4f} (+/- {np.std(cv_recal_scores):.4f})")
    print(f"  Average F1-score:  {np.mean(cv_f1_scores):.4f} (+/- {np.std(cv_f1_scores):.4f})")

    # -------------------------------
    # FINAL MODEL TRAINING (on full training data)
    # -------------------------------
    print("\n[INFO] Training final model on full training data...")
    model = GaussianNB()
    model.fit(X_train_full, y_train_full) # Train final model on the entire training dataset
    print("[INFO] Training completed")

    # -------------------------------
    # PREDICT (on held-out test data)
    # -------------------------------
    print("[INFO] Running prediction on held-out test data...")
    y_pred = model.predict(X_test)

    # -------------------------------
    # EVALUATE (final evaluation)
    # -------------------------------
    print("\n[RESULT] Final Evaluation on Held-Out Test Data:")
    print("\n[RESULT] Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\n[RESULT] Classification Report:")
    print(classification_report(y_test, y_pred, zero_division=0)) # Added zero_division=0 to handle cases where precision/recall might be zero



[INFO] Loading data...
[INFO] Data Loaded: (3500000, 2)

[INFO] Splitting dataset...
[INFO] Train: (2450000, 2)
[INFO] Test : (1050000, 2)

[INFO] PROCESSING WINDOW SIZE = 50
[INFO] gamma1=5.0, gamma2=5.0, gamma3=4.4

[INFO] Creating TRAIN_FULL dataset...
[INFO] Total windows: 49000
[INFO] TRAIN_FULL: 5000/49000 processed
[INFO] TRAIN_FULL: 10000/49000 processed
[INFO] TRAIN_FULL: 15000/49000 processed
[INFO] TRAIN_FULL: 20000/49000 processed
[INFO] TRAIN_FULL: 25000/49000 processed
[INFO] TRAIN_FULL: 30000/49000 processed
[INFO] TRAIN_FULL: 35000/49000 processed
[INFO] TRAIN_FULL: 40000/49000 processed
[INFO] TRAIN_FULL: 45000/49000 processed
[INFO] Finished TRAIN_FULL dataset

[INFO] Creating TEST dataset...
[INFO] Total windows: 21000
[INFO] TEST: 5000/21000 processed
[INFO] TEST: 10000/21000 processed
[INFO] TEST: 15000/21000 processed
[INFO] TEST: 20000/21000 processed
[INFO] Finished TEST dataset
[INFO] Train (full) shape: (49000, 13)
[INFO] Test shape : (21000, 13)

[INFO] Perf

### Saving and Loading the Model

After training, you might want to save the model to avoid retraining every time you use it. `joblib` is a good choice for this, especially for scikit-learn models.

In [ ]:
import joblib

# Define a filename for the model
model_filename = 'gaussian_nb_model.joblib'

# Save the last trained model (from the loop, `model` variable still holds the last one)
joblib.dump(model, model_filename)
print(f"Model saved to {model_filename}")

Model saved to gaussian_nb_model.joblib


You can then load the model back when needed using `joblib.load()`:

In [ ]:
# List all files in the current directory
!ls -F

gaussian_nb_model.joblib     processed_can_data.csv
normal_reference_values.csv  sample_data/


In [ ]:
# Download the model file
from google.colab import files
files.download('gaussian_nb_model.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Training and Saving a Model for Window Size 100 Explicitly

To ensure we have a model trained specifically for a window size of 100, I will run the model training process for this window size and save it under a new filename.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import KFold
import joblib

# Re-load data and define global parameters to ensure they are in scope
# This section is copied from the initial setup
print("\n[INFO] Re-loading data for specific window size processing...")
df = pd.read_csv("/content/processed_can_data.csv") # Use explicit path
ref_df = pd.read_csv("/content/normal_reference_values.csv") # Use explicit path

df["CAN_ID"] = df["CAN_ID"].astype(str).str.strip()
df["Flag"] = df["Flag"].astype(str).str.strip()

# Parameters (copied for completeness, ensure consistency)
window_sizes = [50, 100, 500, 1000] # Added this back for global context, though only 100 is used here
alpha = 0.02
paper_thresholds = {
    50: {"gamma1": 5.0, "gamma2": 5.0},
    100: {"gamma1": 3.0, "gamma2": 4.3},
    500: {"gamma1": 1.8, "gamma2": 2.5},
    1000: {"gamma1": 1.1, "gamma2": 1.6}
}
gamma3_values = {
    50: 4.4,
    100: 2.8,
    500: 3.2,
    1000: 3.7
}

# Split dataset again
split_idx = int(0.7 * len(df))
train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]
print("[INFO] Data re-loaded and split.")

# Ensure utility functions are also defined or globally accessible
# Copying these from the first cell (4curp1XCTNeY) to ensure they are in scope
def calculate_entropy(values):
    counts = Counter(values)
    total = len(values)
    probs = [c/total for c in counts.values()]
    return -sum(p*np.log2(p) for p in probs if p > 0)

def build_graph(can_ids):
    G = nx.DiGraph()
    for i in range(len(can_ids)-1):
        G.add_edge(can_ids[i], can_ids[i+1])
    return G

def graph_features(can_ids):
    G = build_graph(can_ids)

    nodes = G.number_of_nodes()
    edges = G.number_of_edges()

    in_deg = dict(G.in_degree())
    out_deg = dict(G.out_degree())

    max_in = max(in_deg.values()) if in_deg else 0
    min_in = min(in_deg.values()) if in_deg else 0
    max_out = max(out_deg.values()) if out_deg else 0
    min_out = min(out_deg.values()) if out_deg else 0

    pr = nx.pagerank(G) if nodes > 0 else {}
    pr_vals = list(pr.values()) if pr else [0]

    pr_max = max(pr_vals)
    pr_min = min(pr_vals)
    pr_median = np.median(pr_vals)

    return [
        nodes, edges,
        max_in, max_out,
        min_in, min_out,
        pr_max, pr_min, pr_median
    ]

def stat_features(window, ref, gamma1, gamma2, gamma3):
    can_ids = window["CAN_ID"].tolist()

    freq = Counter(can_ids).values()

    mean = np.mean(list(freq))
    std = np.std(list(freq))
    entropy = calculate_entropy(can_ids)

    mean_score = abs(mean - ref["mu_a"]) / (ref["sigma_a"] + 1e-9)
    std_score  = abs(std  - ref["mu_s"]) / (ref["sigma_s"] + 1e-9)
    ent_score  = abs(entropy - ref["mu_e"]) / (ref["sigma_e"] + 1e-9)

    v1 = int(mean_score > gamma1)
    v2 = int(std_score > gamma2)
    v3 = int(ent_score > gamma3)

    score = v1 + v2 + v3

    return mean, std, entropy, score


# Explicitly train and save a model for window size 100

w_100 = 100

print(f"\n{'='*60}")
print(f"[INFO] PROCESSING WINDOW SIZE = {w_100} FOR EXPLICIT SAVE")
print(f"{'='*60}")

# Get reference values and thresholds for w=100
ref_100 = ref_df[ref_df["Window_Size"] == w_100].iloc[0]
gamma1_100 = paper_thresholds[w_100]["gamma1"]
gamma2_100 = paper_thresholds[w_100]["gamma2"]
gamma3_100 = gamma3_values[w_100]

print(f"[INFO] gamma1={gamma1_100}, gamma2={gamma2_100}, gamma3={gamma3_100}")

# Define create_dataset function locally or ensure it's in scope
def create_dataset_w100(data, name):
    print(f"\n[INFO] Creating {name} dataset for w={w_100}...")

    X = []
    y = []

    total_windows = len(data) // w_100
    print(f"[INFO] Total windows: {total_windows}")

    counter = 0

    for start in range(0, len(data), w_100):
        end = start + w_100
        if end > len(data):
            break

        window = data.iloc[start:end]

        attack_ratio = (window["Flag"] == "T").sum() / w_100
        label = 1 if attack_ratio >= alpha else 0

        mean, std, entropy, score = stat_features(
            window, ref_100, gamma1_100, gamma2_100, gamma3_100
        )

        g_feats = graph_features(window["CAN_ID"].tolist())

        features = [mean, std, entropy, score] + g_feats

        X.append(features)
        y.append(label)

        counter += 1

        if counter % 5000 == 0:
            print(f"[INFO] {name}: {counter}/{total_windows} processed")

    print(f"[INFO] Finished {name} dataset for w={w_100}")

    return np.array(X), np.array(y)

X_train_full_100, y_train_full_100 = create_dataset_w100(train_df, "TRAIN_FULL_W100")
X_test_100, y_test_100 = create_dataset_w100(test_df, "TEST_W100")

print(f"[INFO] Train (full w=100) shape: {X_train_full_100.shape}")
print(f"[INFO] Test (w=100) shape : {X_test_100.shape}")

print("\n[INFO] Training model for window size 100...")
model_w100 = GaussianNB()
model_w100.fit(X_train_full_100, y_train_full_100)
print("[INFO] Training completed for window size 100")

model_filename_w100 = 'gaussian_nb_model_w100.joblib'
joblib.dump(model_w100, model_filename_w100)
print(f"Model for window size 100 saved to {model_filename_w100}")


[INFO] PROCESSING WINDOW SIZE = 100 FOR EXPLICIT SAVE
[INFO] gamma1=3.0, gamma2=4.3, gamma3=2.8

[INFO] Creating TRAIN_FULL_W100 dataset for w=100...
[INFO] Total windows: 24500
[INFO] TRAIN_FULL_W100: 5000/24500 processed
[INFO] TRAIN_FULL_W100: 10000/24500 processed
[INFO] TRAIN_FULL_W100: 15000/24500 processed
[INFO] TRAIN_FULL_W100: 20000/24500 processed
[INFO] Finished TRAIN_FULL_W100 dataset for w=100

[INFO] Creating TEST_W100 dataset for w=100...
[INFO] Total windows: 10500
[INFO] TEST_W100: 5000/10500 processed
[INFO] TEST_W100: 10000/10500 processed
[INFO] Finished TEST_W100 dataset for w=100
[INFO] Train (full w=100) shape: (24500, 13)
[INFO] Test (w=100) shape : (10500, 13)

[INFO] Training model for window size 100...
[INFO] Training completed for window size 100
Model for window size 100 saved to gaussian_nb_model_w100.joblib


### Verify Saved Models

Now, let's list the files again to see both models.

In [ ]:
!ls -F

gaussian_nb_model.joblib       normal_reference_values.csv  sample_data/
gaussian_nb_model_w100.joblib  processed_can_data.csv


### Download Model for Window Size 100

Finally, I'll download the newly created model file `gaussian_nb_model_w100.joblib`.

In [ ]:
# Download the model file for window size 100
from google.colab import files
files.download('gaussian_nb_model_w100.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>